In [3]:
%autoreload 2

In [2]:
%reload_ext autoreload
import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime as dt, timedelta

sys.path.append(r'C://Users//Zichen//anaconda3//envs//2ptank//Lib//site-packages//')

In [12]:
def simple_preprocessing_pstim(path):
    #gather all the stimulus.hdf file
    stims ="C://Data//Imaging//260402_overlap//overlap_stim_260402.hdf"#path[:path.rfind('//', 0, -10)] + '//' + mama_stimname#-2 instead of -10
    stims = pd.read_hdf(stims)
    print(path)
    #gather the individual fish stimulus file
    with os.scandir(path) as entries:
        for entry in entries:
            print(entry)
            if 'metadata' in entry.name:
                metadata_path = entry.path
            if 'pstim' in entry.name:
                file = open(entry)
                pstim = file.read()
                file.close()
    
    #from the tail tracking metadata get the start time
    file = open(metadata_path)
    metadata = file.read()
    file.close()
    start_time = metadata[metadata.find('t_protocol_start": "'):].split('"')[2]
    start_time = dt.strptime(start_time, '%Y-%m-%dT%H:%M:%S.%f')
    print(start_time)
    running_stimstart_time = []
    for i in pstim.split("\n")[1:]:
        time = start_time + timedelta(seconds = float(i.split('_')[0]))
        running_stimstart_time = running_stimstart_time + [time]
    if len(stims) == len(running_stimstart_time):
        stims.loc[:, 'real_starttime'] = running_stimstart_time
    else:
        stims.loc[:, 'real_starttime'] = running_stimstart_time + [np.nan] * (len(stims) - len(running_stimstart_time))
    
    #save file
    stims.to_hdf(path + '//stimulus_df.hdf', key = 'stimulus', mode = 'w')

In [18]:
mama_path = 'C://Data//Imaging//260402_overlap//fish5_2'
for fish in np.arange(7, 8):
    path = mama_path  + '//tail//'#+  'fish' + str(fish) #+ '//'
    #preprocessing_pstim(path,)
    simple_preprocessing_pstim(path)

C://Data//Imaging//260402_overlap//fish5_2//tail//
<DirEntry '122730_behavior_log.csv'>
<DirEntry '122730_estimator_log.csv'>
<DirEntry '122730_img.png'>
<DirEntry '122730_metadata.json'>
<DirEntry 'tail_df.hdf'>
<DirEntry '_2_pstim.txt'>
2026-04-04 12:28:17.724512


C:\Users\Zichen\AppData\Local\Temp\ipykernel_39992\128992560.py:34: PerformanceWarning: 
your performance may suffer as PyTables will pickle object types that it cannot
map directly to c-types [inferred_type->mixed-integer,key->block0_values] [items->Index(['stim_name', 'angle', 'velocity', 'stim_type', 'stationary_time',
       'texture', 'circle_center', 'circle_radius', 'angular_velocity'],
      dtype='object')]

  stims.to_hdf(path + '//stimulus_df.hdf', key = 'stimulus', mode = 'w')
